# Results

Walks through every metric in Section 7 of the design, from the files the training and
evaluation scripts write. Run from the repository root after:

```
python -m train.build_features
python -m train.train_models
python -m baselines.hog_svm && python -m baselines.cnn_patch   # optional
python -m eval.evaluate && python -m eval.plots
```

In [ ]:
import json, os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import pandas as pd
from IPython.display import Image, Markdown, display
from src.config import DataPaths

dp, reports = DataPaths(), Path("reports")
m = json.loads((reports / "metrics.json").read_text())
if m["data"].get("warning"):
    print("WARNING:", m["data"]["warning"])
display(Markdown((reports / "results_table.md").read_text(encoding="utf-8")))

## 1. Dataset

In [ ]:
gt = pd.read_csv(dp.ground_truth)
images = pd.read_csv(dp.images, keep_default_na=False)
display(gt.groupby("crop")[["weight_g", "length_mm"]].describe().round(1))
images.groupby(["set", "case"]).size()

## 2. Calibration and segmentation

In [ ]:
display(pd.DataFrame(m["calibration"]["length_error"]).T)
print("mm/px:", m["calibration"]["mm_per_px"], "| reference IoU:", m["segmentation"]["reference_iou"])
pd.Series(m["segmentation"]["detection_count_accuracy"], name="detection count correct")

## 3. Crop classifier and grouping

In [ ]:
cc = m["crop_classifier"]
display(pd.DataFrame(cc["confusion_singles"]).T.rename_axis("true \ predicted"))
{k: v for k, v in cc.items() if k != "confusion_singles"}

## 4. Weight: volume proxy vs multi-feature model

In [ ]:
display(pd.concat({c: pd.DataFrame(v).T for c, v in m["weight"].items()}))
display(Image(str(reports / "weight_pred_vs_true.png")))

## 5. Pairwise decisions

Accuracy should approach 100% for large differences and fall toward 50% for near-identical
items, which is why the SIMILAR band exists.

In [ ]:
display(pd.concat({k: pd.DataFrame(m["pairwise"][k]["bins"]) for k in ("chosen", "all_item_pairs")}))
display(Image(str(reports / "pairwise_vs_difference.png")))

## 6. Absolute Small / Big, and the raw-image baselines

In [ ]:
sb = m["small_big"]
rows = {k: v for k, v in sb["models"].items()}
for b in ("baseline_hog_svm", "baseline_cnn"):
    if b in sb:
        rows[b] = sb[b]["overall"]
display(pd.DataFrame(rows).T[["accuracy", "f1_big"]])
display(Image(str(reports / "small_big_comparison.png")))

## 7. Stress set and efficiency

In [ ]:
display(pd.DataFrame(m["robustness"]).T)
display(Image(str(reports / "stress_cases.png")))
m["efficiency"]